In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing hemolytic peptide datasets (Abdelbaky et al.)

This notebook builds a curated hemolytic peptide dataset starting from multiple raw CSV files associated with **Abdelbaky et al.**. The goal is to produce a clean, standardized dataset that can be used downstream for benchmarking and machine learning experiments.

- **Toxic effect / endpoint:** hemolytic
- **Source:** Abdelbaky et al.
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads raw datasets** (combined + HemoPI1/2/3 + hlppredfuse + rnnamp).
- **Normalizes sequences** (removes whitespace and common formatting artifacts).
- **Detects potentially modified sequences** (e.g., tokens like `AMD` or `ACT`, or mismatches between `text` and `original_sequence`) and stores them in a separate table.
- **Checks duplicates** by sequence and:
  - keeps unique sequences,
  - collapses duplicates when labels are consistent,
  - flags sequences as errors when duplicates have conflicting labels.
- **Builds metadata** from the project Excel metadata sheet, including dataset size and quality-control counts.
- **Exports outputs**:
  - `processed_hemolytic_dataset.csv` (non-modified sequences, deduplicated),
  - `modified_hemolytic_dataset.csv` (modified sequences, deduplicated),
  - `detected_error_sequences.csv` (conflicting-label duplicates),
  - `metadata.json`.

In [2]:
name_source = "Abdelbaky et al."
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants.
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_combined = pd.read_csv(f"{PATH_INPUT}/{name_source}/combined.csv")
df_hemopi1 = pd.read_csv(f"{PATH_INPUT}/{name_source}/HemoPI1.csv")
df_hemopi2 = pd.read_csv(f"{PATH_INPUT}/{name_source}/HemoPI2.csv")
df_hemopi3 = pd.read_csv(f"{PATH_INPUT}/{name_source}/HemoPI3.csv")
df_hlppredfuse = pd.read_csv(f"{PATH_INPUT}/{name_source}/hlppredfuse.csv")
df_rnnamp = pd.read_csv(f"{PATH_INPUT}/{name_source}/rnnamp.csv")

- Concatenating dataset

In [4]:
df_hemopi = pd.concat(
    [df_hemopi1, df_hemopi2, df_hemopi3],
    ignore_index=True
)

df_data = pd.concat(
    [df_combined, df_hlppredfuse, df_rnnamp],
    ignore_index=True
)

In [5]:
# Hemopi sequence cleanup
df_hemopi = (
    df_hemopi
    .assign(
        sequence=lambda d: (
            d["Sequence"]
            .astype(str)
            .str.replace("b'", "", regex=False)
            .str.replace("'", "", regex=False)
            .str.replace(" ", "", regex=False)
        )
    )
    .rename(columns={"y_model_2cl": "label"})
    [["sequence", "label"]]
)

In [6]:
pattern = r'^(A M D|A C T)\b|\b(A M D|A C T)$'

df_data["is_modified"] = (
    df_data["text"].str.contains(pattern, regex=True, na=False)
    |
    df_data.apply(
        lambda row: (
            pd.notna(row["original_sequence"])
            and pd.notna(row["text"])
            and row["original_sequence"] in row["text"]
        ),
        axis=1
    )
)

/tmp/ipykernel_47496/3070070732.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_data["text"].str.contains(pattern, regex=True, na=False)


In [7]:
modified_df = df_data[df_data["is_modified"]]
modified_df = (
    modified_df
    .rename(columns={"text": "sequence", "labels": "label"})
    .assign(
        sequence=lambda d: d["sequence"].astype(str).str.replace(" ", "", regex=False)
    )
    [["sequence", "label"]]
)

In [8]:
df_data.loc[~df_data["is_modified"], "original_sequence"] = df_data["text"]

In [9]:
df_data = (
    df_data
    .rename(columns={"original_sequence": "sequence", "labels": "label"})
    .assign(
        sequence=lambda d: d["sequence"].astype(str).str.replace(" ", "", regex=False)
    )
    [["sequence", "label"]]
)

In [10]:
df_non_modified = pd.concat(
    [df_hemopi, df_data],
    ignore_index=True
)
df_non_modified.shape

(16995, 2)

- Checking duplicates

In [11]:
df_non_modified.head(5)

,sequence,label
0,GIFGKILGVGKKVLCGLSGVC,1
1,KWKSFLKTFKSLKKTVLHTLLKLISS,1
2,KFFKFFKFF,1
3,LLKKLLKKLLKKLLKK,1
4,CAESCVWIPCTVTALLGCSCSNNVCYNGIP,1


In [12]:
df_non_modified["sequence"].unique().shape

(6161,)

In [13]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_non_modified, group_seq="sequence", sort_key="label")

In [14]:
df_remove_duplicated_mod, df_errors_mod, df_unique_mod = processing_duplicated(modified_df, group_seq="sequence", sort_key="label")

In [15]:
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)
df_full.shape

(5851, 2)

In [16]:
df_full_mod = pd.concat([df_unique_mod, df_remove_duplicated_mod], axis=0)
df_full_mod.shape

(2322, 2)

- Working with metada

In [17]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [18]:
raw_total_sequences = (len(df_combined) + len(df_hemopi1) + len(df_hemopi2)
    + len(df_hemopi3) + len(df_hlppredfuse) + len(df_rnnamp))

In [19]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "number_of_modified_sequences": int(len(df_full_mod)),
    "number_of_erroneous_modified_sequences": int(len(df_errors_mod)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2024, 6, 29, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'csv',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from Swiss-Prot',
 'repository or server': 'https://github.com/mohamedelhakim/Hemolytic-End-to-End-Architecture',
 'publication': 'https://link.springer.com/article/10.1186/s12859-024-05983-4',
 'number_of_raw_sequences': 16995,
 'number_of_sequences_retained': 5851,
 'number_of_positive_sequences': 1983,
 'number_of_negative_sequences': 3868,
 'number_of_erroneous_sequences': 309,
 'number_of_modified_sequences': 2322,
 'number_of_erroneous_modified_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [20]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [21]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_mod.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/modified_hemolytic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)